In [ ]:
# Cell — config provenance record; fill only from the actual team config
import json, hashlib, subprocess
from pathlib import Path
from datetime import datetime, timezone

CONFIG_PATH = Path("/content/a5-adversarial-internals/configs/track3.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Track 3 config not found. Locate the mentor/team config; do not create guessed parameters."
    )
config_bytes = CONFIG_PATH.read_bytes()
config_sha256 = hashlib.sha256(config_bytes).hexdigest()
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_sha = "unknown"

record = {
    "config_path": str(CONFIG_PATH),
    "config_sha256": config_sha256,
    "git_commit": git_sha,
    "recorded_utc": datetime.now(timezone.utc).isoformat(),
}
out = DRIVE_ROOT / "manifests" / "track3_provenance.json"
out.write_text(json.dumps(record, indent=2))
print(record)

FileNotFoundError: Track 3 config not found. Locate the mentor/team config; do not create guessed parameters.

In [ ]:
# Cell — pair-level deterministic 70/30 split (same pair never leaks across splits)
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

pairs = pd.read_parquet("/content/drive/MyDrive/MYSSP/pairs_with_activation_paths.parquet")
if "pair_id" not in pairs:
    raise ValueError("pairs table needs pair_id")
unique_pairs = pairs[["pair_id"]].drop_duplicates().copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=2026)
train_idx, test_idx = next(splitter.split(unique_pairs, groups=unique_pairs["pair_id"]))
unique_pairs["split"] = "test"
unique_pairs.loc[train_idx, "split"] = "train"
split_map = unique_pairs.set_index("pair_id")["split"]
pairs["split"] = pairs["pair_id"].map(split_map)
assert pairs.groupby("pair_id")["split"].nunique().max() == 1
pairs.to_parquet(DRIVE_ROOT / "manifests" / "pair_split.parquet", index=False)
print(pairs["split"].value_counts(), "rows; unique pairs:", pairs["pair_id"].nunique())

In [ ]:
# Cell — judge agreement report from two independent label columns
# Input CSV must contain one row per judged item, with columns: item_id, judge_a, judge_b.
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score
from pathlib import Path

JUDGE_FILE = DRIVE_ROOT / "judges" / "judge_labels.csv"
if not JUDGE_FILE.exists():
    raise FileNotFoundError(f"Expected labels at {JUDGE_FILE}; first run both judges and save labels.")
df = pd.read_csv(JUDGE_FILE)
needed = {"item_id", "judge_a", "judge_b"}
if not needed.issubset(df.columns):
    raise ValueError(f"Missing columns: {needed - set(df.columns)}")
df = df.dropna(subset=["judge_a", "judge_b"])
if len(df) == 0:
    raise ValueError("No jointly labelled rows.")
labels = sorted(set(df["judge_a"].astype(str)) | set(df["judge_b"].astype(str)))
report = {
    "n_joint": int(len(df)),
    "raw_agreement": float(accuracy_score(df["judge_a"].astype(str), df["judge_b"].astype(str))),
    "cohen_kappa": float(cohen_kappa_score(df["judge_a"].astype(str), df["judge_b"].astype(str))),
    "labels": labels,
    "confusion_matrix": confusion_matrix(df["judge_a"].astype(str), df["judge_b"].astype(str), labels=labels).tolist(),
}
out = DRIVE_ROOT / "judges" / "judge_agreement_report.json"
out.write_text(__import__("json").dumps(report, indent=2))
print(report)
print("Saved:", out)